# VQA Engine Test & Pipeline Demos

This notebook tests the VQAEngine and demonstrates various LangChain pipelines for multimodal processing:

## Pipeline Demos
1. **Basic VQA** - Direct engine usage
2. **Runnable Pipeline** - Using LangChain's `|` operator
3. **Batch Processing** - Process multiple images efficiently
4. **Parallel Analysis** - Concurrent multi-aspect analysis
5. **Streaming** - Real-time response streaming
6. **Conditional Routing** - Question-type based routing

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
from dotenv import load_dotenv
load_dotenv('../.env')

In [ ]:
from src.model import VQAEngine
from PIL import Image
import io
import base64
import time
import logging
import hashlib
import asyncio
import re
from typing import List, Dict, Any, Optional
from functools import lru_cache

from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

## Setup: Create Test Image

In [ ]:
def encode_image(image: Image.Image) -> str:
    """Convert PIL Image to base64 string."""
    buffered = io.BytesIO()
    image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# Create test image
img = Image.new('RGB', (100, 100), color='red')
img_b64 = encode_image(img)
display(img)

## 1. Basic VQA

In [ ]:
engine = VQAEngine()
question = "What color is this image?"
answer = engine.predict(img_b64, question)
print(f"Q: {question}")
print(f"A: {answer}")

## 2. Runnable Pipeline

In [ ]:
def prepare_multimodal_input(data: Dict[str, str]) -> Dict[str, Any]:
    return {
        "messages": [
            HumanMessage(
                content=[
                    {"type": "text", "text": data["question"]},
                    {"type": "image_url", "image_url": f"data:image/jpeg;base64,{data['image_b64']}"}
                ]
            )
        ]
    }

def parse_response(data: Dict) -> str:
    return data["response"].content

prepare_step = RunnableLambda(prepare_multimodal_input)
llm_step = RunnableLambda(lambda d: {"response": ChatGoogleGenerativeAI(model="gemini-2.5-flash").invoke(d["messages"])})
parse_step = RunnableLambda(parse_response)
vqa_pipeline = prepare_step | llm_step | parse_step

result = vqa_pipeline.invoke({"image_b64": img_b64, "question": "Describe this image"})
print(f"Pipeline Result: {result}")

## 3. Batch Processing

In [ ]:
def batch_vqa(inputs: List[Dict]) -> List[str]:
    """Process multiple VQA requests with timing."""
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
    results = []
    start = time.time()
    
    for item in inputs:
        try:
            msg = HumanMessage(content=[
                {"type": "text", "text": item["question"]},
                {"type": "image_url", "image_url": f"data:image/jpeg;base64,{item['image_b64']}"}
            ])
            resp = llm.invoke([msg])
            results.append(resp.content)
        except Exception as e:
            results.append(f"Error: {str(e)}")
    
    elapsed = time.time() - start
    print(f"Processed {len(inputs)} in {elapsed:.2f}s ({len(inputs)/elapsed:.2f} img/sec)")
    return results

test_images = [
    {"image_b64": encode_image(Image.new('RGB', (100, 100), c)), "question": "What color?"}
    for c in ['red', 'blue', 'green']
]
batch_results = batch_vqa(test_images)

## 4. Parallel Analysis (RunnableParallel)

In [ ]:
def color_analyzer(data: Dict) -> str:
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
    msg = HumanMessage(content=[
        {"type": "text", "text": "Name the dominant color only."},
        {"type": "image_url", "image_url": f"data:image/jpeg;base64,{data['image_b64']}"}
    ])
    return llm.invoke([msg]).content

def object_analyzer(data: Dict) -> str:
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
    msg = HumanMessage(content=[
        {"type": "text", "text": "What shapes/objects?"},
        {"type": "image_url", "image_url": f"data:image/jpeg;base64,{data['image_b64']}"}
    ])
    return llm.invoke([msg]).content

parallel_analysis = RunnableParallel(
    color=RunnableLambda(color_analyzer),
    objects=RunnableLambda(object_analyzer)
)

results = parallel_analysis.invoke({"image_b64": img_b64})
print(f"🎨 Color: {results['color']}")
print(f"📦 Objects: {results['objects']}")

## 5. Streaming Pipeline

In [ ]:
def streaming_vqa(image_b64: str, question: str):
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", streaming=True)
    msg = HumanMessage(content=[
        {"type": "text", "text": question},
        {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_b64}"}
    ])
    print("Streaming:")
    for chunk in llm.stream([msg]):
        print(chunk.content, end="", flush=True)
    print()

streaming_vqa(img_b64, "Describe this image")

## 6. Conditional Pipeline

In [ ]:
def route_by_question(data: Dict) -> str:
    q = data["question"].lower()
    if "color" in q:
        return "color"
    elif "describe" in q:
        return "describe"
    return "general"

def conditional_vqa(data: Dict) -> str:
    route = route_by_question(data)
    print(f"[Routed to: {route}]")
    
    handlers = {
        "color": lambda d: f"[COLOR] {VQAEngine().predict(d['image_b64'], d['question'])}",
        "describe": lambda d: f"[DESC] {VQAEngine().predict(d['image_b64'], d['question'])}",
        "general": lambda d: f"[GEN] {VQAEngine().predict(d['image_b64'], d['question'])}"
    }
    return handlers.get(route, handlers["general"])(data)

for q in ["What color?", "Describe this", "Is it square?"]:
    print(f"\nQ: {q}")
    print(conditional_vqa({"image_b64": img_b64, "question": q}))

---

# 🛠 Production Improvements (from improve.md)

The following sections demonstrate patterns from `improve.md` for production readiness.

## 7. Timeout Handling Pattern

**Reference:** improve.md Section 3

Demonstrate timeout patterns for LLM calls.

In [ ]:
import signal
from contextlib import contextmanager

class TimeoutError(Exception):
    pass

@contextmanager
def timeout(seconds: int):
    """Context manager for timeout (Unix-only for signal)."""
    def handler(signum, frame):
        raise TimeoutError(f"Request timed out after {seconds}s")
    
    # Set the alarm (Unix only)
    old_handler = signal.signal(signal.SIGALRM, handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

class VQAEngineWithTimeout(VQAEngine):
    """Extended engine with timeout support."""
    
    def predict_with_timeout(self, image_b64: str, text: str, timeout_sec: int = 10) -> str:
        """Predict with timeout protection."""
        try:
            with timeout(timeout_sec):
                return self.predict(image_b64, text)
        except TimeoutError as e:
            return f"Error: {str(e)}"
        except Exception as e:
            return f"Error: {str(e)}"

# Demo
engine_timeout = VQAEngineWithTimeout()
result = engine_timeout.predict_with_timeout(img_b64, "What color?", timeout_sec=30)
print(f"Result with timeout: {result}")

## 8. Response Caching

**Reference:** improve.md Section 8

Demonstrate in-memory response caching to reduce API costs.

In [ ]:
class VQAEngineWithCache(VQAEngine):
    """Engine with simple in-memory caching."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._cache = {}
        self._cache_hits = 0
    
    def _get_cache_key(self, image_b64: str, text: str) -> str:
        """Generate cache key from inputs."""
        combined = f"{image_b64}:{text}"
        return hashlib.md5(combined.encode()).hexdigest()
    
    def predict(self, image_b64: str, text: str) -> str:
        """Predict with caching."""
        cache_key = self._get_cache_key(image_b64, text)
        
        if cache_key in self._cache:
            self._cache_hits += 1
            print(f"[CACHE HIT #{self._cache_hits}] Serving from cache")
            return self._cache[cache_key]
        
        print("[CACHE MISS] Calling LLM API...")
        result = super().predict(image_b64, text)
        self._cache[cache_key] = result
        return result
    
    def get_stats(self) -> Dict:
        """Return cache statistics."""
        return {
            "cache_size": len(self._cache),
            "cache_hits": self._cache_hits,
            "hit_rate": self._cache_hits / (len(self._cache) + self._cache_hits) if self._cache_hits > 0 else 0
        }

# Demo
cache_engine = VQAEngineWithCache()

# First call - cache miss
r1 = cache_engine.predict(img_b64, "What color is this?")
print(f"First: {r1}\n")

# Second call - cache hit
r2 = cache_engine.predict(img_b64, "What color is this?")
print(f"Second: {r2}\n")

print(f"Cache stats: {cache_engine.get_stats()}")

## 9. Input Validation

**Reference:** improve.md Section 7

Demonstrate question validation and sanitization.

In [ ]:
class ValidationError(Exception):
    pass

def validate_question(question: str) -> tuple[bool, Optional[str]]:
    """Validate question input. Returns (is_valid, error_message)."""
    
    # Check empty
    if not question or not question.strip():
        return False, "Question cannot be empty"
    
    # Check length
    if len(question) > 500:
        return False, "Question too long (max 500 characters)"
    
    # Check for suspicious characters
    if re.search(r'[<>{}]', question):
        return False, "Question contains invalid characters"
    
    # Check for blocked words (example)
    blocked = ["hack", "exploit", "ignore previous"]
    if any(word in question.lower() for word in blocked):
        return False, "Question contains prohibited content"
    
    return True, None

def sanitize_question(question: str) -> str:
    """Clean and normalize question."""
    # Strip whitespace
    cleaned = question.strip()
    # Remove extra spaces
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned

class VQAEngineWithValidation(VQAEngine):
    """Engine with input validation."""
    
    def predict(self, image_b64: str, text: str) -> str:
        # Validate first
        is_valid, error = validate_question(text)
        if not is_valid:
            return f"Validation Error: {error}"
        
        # Sanitize
        clean_text = sanitize_question(text)
        return super().predict(image_b64, clean_text)

# Test validation
val_engine = VQAEngineWithValidation()

test_questions = [
    ("What color?", True),
    ("", False),
    ("x" * 600, False),
    ("What is this?", True),
]

for q, should_work in test_questions:
    result = val_engine.predict(img_b64, q)
    status = "✅" if (not "Validation Error" in result) == should_work else "❌"
    print(f"{status} '{q[:50]}...' -> {result[:60]}")

## 10. Structured Logging

**Reference:** improve.md Section 6

Replace print() with proper Python logging.

In [ ]:
import logging
import sys

# Configure structured logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - [%(filename)s:%(lineno)d] - %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("vqa_engine.log")  # Also log to file
    ]
)

logger = logging.getLogger("vqa_engine_improved")

class VQAEngineWithLogging(VQAEngine):
    """Engine with structured logging."""
    
    def __init__(self, model_name="gemini-2.5-flash"):
        logger.info(f"Initializing VQAEngine with model={model_name}")
        super().__init__(model_name)
        self.request_count = 0
    
    def predict(self, image_b64: str, text: str) -> str:
        self.request_count += 1
        request_id = self.request_count
        
        logger.info(f"[Request {request_id}] Starting prediction", extra={
            "request_id": request_id,
            "question_length": len(text),
            "image_size": len(image_b64)
        })
        
        start_time = time.time()
        try:
            result = super().predict(image_b64, text)
            elapsed = time.time() - start_time
            
            logger.info(f"[Request {request_id}] Success in {elapsed:.2f}s", extra={
                "request_id": request_id,
                "latency_ms": elapsed * 1000,
                "result_length": len(result)
            })
            return result
            
        except Exception as e:
            elapsed = time.time() - start_time
            logger.error(f"[Request {request_id}] Failed after {elapsed:.2f}s", extra={
                "request_id": request_id,
                "error": str(e),
                "latency_ms": elapsed * 1000
            }, exc_info=True)
            return f"Error: {str(e)}"

# Demo
log_engine = VQAEngineWithLogging()
result = log_engine.predict(img_b64, "Test logging")
print(f"\nPrediction completed. Check vqa_engine.log file.")

## 11. Async Patterns

**Reference:** improve.md Section 11

Demonstrate async/await for better concurrency.

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

class AsyncVQAEngine(VQAEngine):
    """Engine with async support."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._executor = ThreadPoolExecutor(max_workers=4)
    
    async def predict_async(self, image_b64: str, text: str) -> str:
        """Async version of predict."""
        loop = asyncio.get_event_loop()
        # Run blocking call in executor
        result = await loop.run_in_executor(
            self._executor,
            self.predict,
            image_b64,
            text
        )
        return result
    
    async def batch_predict_async(self, items: List[Dict]) -> List[str]:
        """Process multiple items concurrently."""
        tasks = [
            self.predict_async(item["image_b64"], item["question"])
            for item in items
        ]
        return await asyncio.gather(*tasks)

async def demo_async():
    """Demo async processing."""
    engine = AsyncVQAEngine()
    
    items = [
        {"image_b64": encode_image(Image.new('RGB', (100, 100), c)), "question": "What color?"}
        for c in ['red', 'blue', 'green', 'yellow']
    ]
    
    print("Processing 4 images concurrently...")
    start = time.time()
    results = await engine.batch_predict_async(items)
    elapsed = time.time() - start
    
    print(f"Completed in {elapsed:.2f}s")
    for i, result in enumerate(results):
        print(f"  {i+1}. {result}")

# Run async demo
await demo_async()

## 12. Health Check Pattern

**Reference:** improve.md Section 5

Demonstrate health check implementation for monitoring.

In [ ]:
import json
from datetime import datetime

class HealthChecker:
    """Health check utilities for VQA service."""
    
    def __init__(self, engine: VQAEngine):
        self.engine = engine
        self.start_time = datetime.now()
        self.request_count = 0
        self.error_count = 0
    
    def health(self) -> Dict:
        """Liveness check."""
        return {
            "status": "healthy",
            "timestamp": datetime.now().isoformat(),
            "version": "0.1.0"
        }
    
    def ready(self) -> Dict:
        """Readiness check - verifies engine is functional."""
        try:
            # Quick smoke test with tiny image
            test_img = encode_image(Image.new('RGB', (10, 10), 'white'))
            _ = self.engine.predict(test_img, "test")
            return {
                "ready": True,
                "engine": "initialized",
                "timestamp": datetime.now().isoformat()
            }
        except Exception as e:
            return {
                "ready": False,
                "error": str(e),
                "timestamp": datetime.now().isoformat()
            }
    
    def metrics(self) -> Dict:
        """Service metrics."""
        uptime = (datetime.now() - self.start_time).total_seconds()
        return {
            "uptime_seconds": uptime,
            "total_requests": self.request_count,
            "error_count": self.error_count,
            "error_rate": self.error_count / max(self.request_count, 1),
            "status": "operational"
        }

# Demo
checker = HealthChecker(VQAEngine())

print("=== Health Check ===" )
print(json.dumps(checker.health(), indent=2))

print("\n=== Readiness Check ===" )
print(json.dumps(checker.ready(), indent=2))

print("\n=== Metrics ===" )
print(json.dumps(checker.metrics(), indent=2))

---

## Summary

This notebook demonstrates:

### Pipeline Patterns
1. **Basic VQA** - Direct engine usage
2. **Runnable Pipelines** - Composable chains with `|` operator
3. **Batch Processing** - Efficient multi-image handling
4. **Parallel Execution** - Concurrent multi-aspect analysis
5. **Streaming** - Real-time token streaming
6. **Conditional Routing** - Question-type based routing

### Production Improvements (improve.md)
7. **⏱ Timeout Handling** - Prevent hanging requests
8. **💾 Response Caching** - Reduce API costs
9. **✅ Input Validation** - Security and data quality
10. **📝 Structured Logging** - Observability
11. **🔄 Async Patterns** - Better concurrency
12. **❤️ Health Checks** - Production monitoring

To apply these improvements to the actual codebase, copy the class implementations from this notebook to the corresponding files in `src/`.